# 8.5 Final Model Selection and Deployment

## Course 3: Machine Learning for Higher Education — Advanced Applications

The institution has always utilized survey data to get the pulse of student interests, preferences and priorities. Now they realize that survey data can also be used to enhance unsupervised and supervised machine learning models. In this notebook, we'll recap the model selection journey and deploy the survey-enhanced xgboost model to new data.

## Learning Objectives

1.  **Understand** the criteria for selecting a final machine learning model for deployment, moving beyond mere accuracy to consider actionability, explainability, and institutional fit.
2.  **Learn** the end-to-end process of deploying a machine learning model, encompassing data preparation, prediction generation, and the application of practical outreach thresholds.
3.  **Develop** effective communication strategies for presenting model outputs to diverse stakeholders, including the creation of interpretable risk bands and advisor-facing outreach lists.
4.  **Identify** crucial considerations for responsible AI deployment, such as addressing ethical implications, ensuring privacy, establishing human oversight, and planning for ongoing monitoring.
5.  **Recognize** the essential components of a robust deployable model, including the model artifact, feature schema, model card, and a comprehensive deployment checklist.

## 1. Final Model Selection Logic

Recall that the advanced comparison in **8.4** compares four models:

| Model | Feature Set | Main Strength |
|---|---|---|
| Regularized Logistic Regression | Administrative/student-record features | Most interpretable and easiest to explain |
| Random Forest | Administrative/student-record features | Strong nonlinear baseline with good robustness |
| XGBoost | Administrative/student-record features | Strong predictive performance on tabular data |
| Survey-Enhanced XGBoost | Academic + demographic + survey/text features | Best multimodal model because it uses both records and student voice |

For this final deployment notebook, the **recommended best overall model** is:

## Survey-Enhanced XGBoost

This recommendation is based on the logic developed across Modules 8.1–8.4:

- 8.1 established why survey data can add student experience signals that administrative records miss.
- 8.2 showed how survey/text features can reveal student personas and hidden patterns.
- 8.3 trained a survey-enhanced XGBoost model for 3rd-semester status.
- 8.4 compared that model against the original 4.1 model families.

The deployment question is not simply, **Which model has the highest accuracy?**  
The stronger question is, **Which model provides the best balance of prediction, actionability, and institutional explanation?**

## 2. Setup

Expected files:

- `training.csv`
- `testing.csv`
- `ML_SURVEY_MASTER_TRAIN.csv`
- `ML_SURVEY_MASTER_TEST.csv`

The two survey master files are the preferred deployment inputs because they include the engineered survey and text features from Module 7 and Module 8.

In [ ]:
# If needed in Colab, uncomment the next line:
# !pip -q install xgboost

import numpy as np
import pandas as pd
import warnings
import time
import joblib
import json
from pathlib import Path

warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report, brier_score_loss, log_loss
)

import pickle

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 3. Data Loading Helper

The original Course 3 notebooks sometimes use `../data/`, while the Module 8 notebooks often use a Google Drive path. The helper below checks several common locations.

Update `candidate_dirs` if your files live somewhere else.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
data_filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'

# Deploy_Survey_Data.csv includes a SID column (student identifier) plus the
# model's feature columns — split them apart before scoring, since the model
# was never trained on a SID column.
df_deploy_full = pd.read_csv(f'{data_filepath}Deploy_Survey_Data.csv')
deploy_sids = df_deploy_full['SID']
df_deploy = df_deploy_full.drop(columns=['SID'])
pd.set_option('display.max_columns', None)
df_deploy

# 4. Pickle in the Best Model

In [14]:
# Load the model from the pickle file
model_filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Models/'
filename1 = f'{model_filepath}Survey_xgb_model.pkl'
survey_xgb_model = pickle.load(open(filename1, 'rb'))

filename2 = f'{model_filepath}kmeans_model.pkl'
kmeans_cluster_model = pickle.load(open(filename2, 'rb'))
# You can now use the loaded_model_pkl object for predictions or other tasks
print("Model loaded successfully from pickle file.")

Model loaded successfully from pickle file.


## 5. Deploy the Recommended Model to the Hold-Out Set

In a real institutional workflow, the hold-out set represents students the model did **not** see during training.

The deployment steps are:

1. Load the saved model and feature schema.
2. Generate a predicted probability for each student.
3. Convert probabilities into risk bands.
4. Share a carefully framed outreach list with advisors or student success teams.

In [ ]:
# Predicted probability of departure for each hold-out student
holdout_prob = survey_xgb_model.predict_proba(df_deploy)[:, 1]

# Create an indicator variable that =1 for students predicted to depart and =0 if not
holdout_pred_default = (holdout_prob >= 0.50).astype(int)

holdout_scores = df_deploy.copy()
holdout_scores.insert(0, 'SID', deploy_sids.values)
holdout_scores['departure_risk_score'] = holdout_prob
holdout_scores['predicted_departed_default_050'] = holdout_pred_default

holdout_scores

Let's take a look at some sumary statistics around departure for our deployment set:

In [ ]:
holdout_scores[['departure_risk_score', 'predicted_departed_default_050']].describe().round(4)

## 6. Choose a Practical Outreach Threshold

A probability threshold of `0.50` is mathematically simple, but it may not match advising capacity.

For deployment, institutions often choose a threshold based on one of these practical rules:

| Threshold Strategy | Meaning |
|---|---|
| Default threshold | Flag students with predicted probability ≥ 0.50 |
| Capacity threshold | Flag the top X% of students because that is how many advisors can contact |
| Recall-oriented threshold | Lower the threshold to catch more true departures, accepting more false positives |
| Precision-oriented threshold | Raise the threshold so outreach lists are smaller and more concentrated |

Below, we create a **capacity-based threshold** that flags the top 20% of hold-out students by predicted risk. We use the statistical concept of *quantiles* (generalized version of percentiles), which allow us to start with the desired percent and work backwards to find the relevant corresponding threshold.

In [17]:
capacity_share = 0.20  # Change this if advising capacity is larger or smaller.
capacity_threshold = float(np.quantile(holdout_prob, 1 - capacity_share))
holdout_scores['flag_top_20pct_capacity'] = (holdout_scores['departure_risk_score'] >= capacity_threshold).astype(int)

print(f'Capacity-based threshold for top {capacity_share:.0%}: {capacity_threshold:.4f}')
print('Number flagged:', int(holdout_scores['flag_top_20pct_capacity'].sum()))
print('Share flagged:', holdout_scores['flag_top_20pct_capacity'].mean().round(4))


Capacity-based threshold for top 20%: 0.6599
Number flagged: 25
Share flagged: 0.2033


## 7. Convert Risk Scores into Stakeholder-Friendly Risk Bands

Risk bands are easier to communicate than raw probabilities.

The exact labels should be reviewed by campus leadership and advising teams before use. Avoid stigmatizing language. In this notebook, we use:

- **Priority outreach**
- **Monitor/support**
- **Routine support**

These labels focus on the institutional action rather than labeling the student.

In [18]:
def assign_risk_band(score):
    if score >= np.quantile(holdout_prob, 0.80):
        return 'Priority outreach'
    elif score >= np.quantile(holdout_prob, 0.50):
        return 'Monitor/support'
    else:
        return 'Routine support'

holdout_scores['support_band'] = holdout_scores['departure_risk_score'].apply(assign_risk_band)

band_summary = (
    holdout_scores
    .groupby('support_band')
    .agg(
        students=('departure_risk_score', 'size'),
        avg_risk_score=('departure_risk_score', 'mean'),
    )
    .sort_values('avg_risk_score', ascending=False)
)

band_summary.round(4)

,students,avg_risk_score
support_band,,
Priority outreach,25,0.8136
Monitor/support,37,0.4536
Routine support,61,0.2582


In [19]:
fig = px.bar(
    band_summary.reset_index(),
    x='support_band',
    y='students',
    text='students',
    title='Hold-Out Students by Support Band',
    labels={'support_band': 'Support Band', 'students': 'Number of Students'}
)
fig.update_traces(textposition='outside')
fig.update_layout(height=450)
fig.show()

## 8. Create an Advisor-Facing Outreach List

The outreach list should contain only what advisors need to act. In production, avoid sharing unnecessary sensitive features.

Recommended columns:

- A student identifier
- Risk score
- Support band
- A few high-level academic indicators
- A note that the score is advisory and should not be used punitively

In [ ]:
# Deploy_Survey_Data.csv now includes a real SID column, so holdout_scores
# already has each student's risk score correctly attached to their SID —
# no join against a separate names file is needed.
df_deploy_risk = holdout_scores

In [ ]:

recommended_context_cols = [
    'SID',
    'departure_risk_score',
    'support_band',
    'flag_top_20pct_capacity',
    'HS_GPA',
    'GPA_1',
    'GPA_2',
    'DFW_RATE_1',
    'DFW_RATE_2',
    'UNITS_ATTEMPTED_1',
    'UNITS_ATTEMPTED_2'
]



advisor_outreach_list = (
    df_deploy_risk[recommended_context_cols]
    .sort_values('departure_risk_score', ascending=False)
    .reset_index(drop=True)
)

advisor_outreach_list.head(123).round(4)

## 9. Save the Model, Feature Schema, and Deployment Outputs

A deployable model is more than the model file. It should include:

- The model artifact
- The feature list/schema
- The risk scoring output
- A model card
- A note explaining the intended use and limitations

## 10. Model Card for Stakeholders

A deployable model is more than the model file. It should include:

- The model artifact
- The feature list/schema
- The risk scoring output
- A model card
- A note explaining the intended use and limitations

The following model card documents what the model is, what it is for, and what its limits are.

| Field | Description |
|:------|:-----------|
| **Model Name** | Survey-Enhanced Student Departure Risk Model v1.0 |
| **Model Type** | XGBoost binary classifier |
| **Task** | Binary classification: predict 3rd semester departure |
| **Training Data** | CSULB first-time freshmen cohorts |
| **Features** | HS GPA, college GPA, DFW rates, demographics, qualitative survey responses |
| **Performance (AUC)** | 0.8791 |
| **Intended Use** | Early warning system for academic advisors |
| **Limitations** | Trained on CSULB data; may not generalize to other institutions |
| **Ethical Considerations** | Low bias across demographic groups |
| **Retraining Schedule** | Annually, with each new cohort |
| **Owner** | Institutional Research and Analytics Department |

## 11. Plain-Language Explanation for Non-Technical Stakeholders

### What the model does

The model estimates which students in the hold-out set appear more likely to depart before the third semester. It uses information already prepared in the survey master matrix, including academic indicators and survey/text-derived signals.

### What the model does not do

The model does not determine a student's future. It does not explain causation. It should not be used to punish students, restrict opportunities, or make automatic decisions.

### How the institution should use it

The best use is supportive prioritization. Students in the highest support band should be considered for earlier advising, resource connection, or check-in communication.

### How to explain the risk score

A risk score means:

> “Based on patterns from prior students, this student looks similar to students who were more likely to depart. The score is a prompt for supportive outreach, not a judgment about the student.”

## 12. Deployment Checklist

Before using this model operationally, complete the following checklist. Note that some of these checks were performed as part of our model selection in section 8.4:

| Area | Check |
|---|---|
| Technical validation | Confirm model runs on new data without column mismatch errors |
| Hold-out performance | Review F1, recall, ROC-AUC, and precision-recall performance |
| Threshold choice | Align threshold with advising capacity and intervention design |
| Equity review | Compare performance and flag rates across student subgroups |
| Privacy review | Remove unnecessary sensitive fields from outreach files |
| Documentation | Save model card, feature schema, model artifact, and scoring output |
| Human oversight | Make clear that advisors make decisions, not the model |
| Monitoring | Track whether outreach improves outcomes and whether model drift occurs |
| Governance | Review with institutional data governance or analytics leadership |

## 13. Final Recommendation

In this notebook, we deployed the recommended final model - **Survey-Enhanced XGBoost**. This extends the original 4.1 comparison by adding survey and text-derived features to the strongest tree-based modeling family used in the course. This allows the model to use both administrative records and student voice.

The model has clear advantages from the institutional perspective; it produces an actionable risk score that can be translated into support bands for advising teams. The output is not meant to label students; it is meant to help the institution organize limited support capacity. Deploy the model as a **decision-support tool**, not a decision-making system.

## 14. Conclusion

You have learned more than a set of algorithms. You've experienced a powerful framework for creating an end to end decision support system that incorporates student background, experience and voice to strengthen predictive modeling. With this training, you are ready to apply this process to real higher ed challenges in an ethical way.  